# 03 - Preprocessing and Augmentation

**Purpose:** build a reproducible transformation pipeline appropriate to the selected methods and dataset.

**Expected inputs:** audited raw/interim data, annotations, and the approved split strategy.

**Expected outputs:** validated preprocessing, augmentation, label conversion, and model-ready split generation.

**Retain for the report:** before/after examples, transformation rationale, parameter choices, leakage safeguards, and any annotation conversions.

In [ ]:
import csv
from collections import Counter
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd / 'implementation', cwd, *cwd.parents]
IMPLEMENTATION_ROOT = next((p for p in candidates if p.name == 'implementation' and (p / 'stages').exists()), None)
if IMPLEMENTATION_ROOT is None:
    raise RuntimeError('Start Jupyter from the repository root or implementation directory.')
INTERIM_DATA = IMPLEMENTATION_ROOT / 'data' / 'interim'
PROCESSED_DATA = IMPLEMENTATION_ROOT / 'data' / 'processed'

## Pipeline definition

Create the model-ready relevant-annotation manifest without modifying raw data. Fit data-dependent transformations on the training split only and apply deterministic evaluation transforms to internal validation and locked test data.

In [ ]:
source_manifest = INTERIM_DATA / 'manifests' / 'fashionpedia-annotations.csv'
if not source_manifest.exists():
    raise FileNotFoundError('Run stage 02 before preprocessing.')

PROCESSED_DATA.mkdir(parents=True, exist_ok=True)
destination = PROCESSED_DATA / 'relevant-fashionpedia-annotations.csv'
split_counts = Counter()
with source_manifest.open('r', encoding='utf-8', newline='') as source_handle, destination.open('w', encoding='utf-8', newline='') as destination_handle:
    reader = csv.DictReader(source_handle)
    writer = csv.DictWriter(destination_handle, fieldnames=reader.fieldnames)
    writer.writeheader()
    for row in reader:
        if row['relevant'] != '1':
            continue
        writer.writerow(row)
        split_counts[row['final_split']] += 1

print(f'Wrote {destination}')
print(dict(split_counts))
print('Feature extraction must consume this manifest in batches and store float32 caches outside Git.')
